# 🎬 Movie Review Sentiment Analysis — PREDICT FILE v3
**Purpose:** Load the saved `.pkl` model from Drive and predict sentiment on new reviews  
**Classes:** Highly Positive | Positive | Neutral | Negative | Highly Negative  
**Requires:** Run `train_3.ipynb` first to generate the `.pkl` files in `Movie_Sentiment_Analysis/train/`

### ✨ What's new vs predict_2.ipynb:
- **Loads `sentiment_model_v3.pkl`** — the stacking ensemble from train_3
- **Contraction expansion** — `can't` → `cannot`, `isn't` → `is not` (matches train_3 preprocessing)
- **VADER sentiment scores** — 4 numeric features appended to TF-IDF (compound, pos, neg, neu)
- **Handcrafted features** — review length, exclamation count, caps ratio, etc.
- **Combined feature matrix** — `[Word TF-IDF | Char TF-IDF | VADER | Handcrafted]` exactly as trained
- **ShiftToNonNegative** — same transformer used inside the stacking ensemble

In [ ]:
# ── Cell 1 — Install & Imports ────────────────────────────────────────────────
!pip install nltk scikit-learn vaderSentiment contractions -q

import nltk
for pkg in ['stopwords', 'punkt', 'punkt_tab', 'wordnet', 'vader_lexicon']:
    nltk.download(pkg, quiet=True)

import pickle
import os
import string
import re
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from scipy.sparse import hstack, csr_matrix
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

try:
    import contractions
    HAS_CONTRACTIONS = True
except ImportError:
    HAS_CONTRACTIONS = False

print('✅ Imports done.')

In [ ]:
# ── Cell 2 — Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR    = '/content/drive/MyDrive/Movie_Sentiment_Analysis'
TRAIN_DIR   = os.path.join(BASE_DIR, 'train')
PREDICT_DIR = os.path.join(BASE_DIR, 'predict')
os.makedirs(PREDICT_DIR, exist_ok=True)

print('✅ Google Drive mounted.')
print(f'   Looking for model in: {TRAIN_DIR}')

In [ ]:
# ── Cell 3 — Load Saved Model & BOTH TF-IDF Vectorizers from Drive ────────────
# train_3 saves THREE files:
#   sentiment_model_v3.pkl        — the best model (Stacking Ensemble)
#   tfidf_word_vectorizer_v3.pkl  — word-level TF-IDF (fitted on training data)
#   tfidf_char_vectorizer_v3.pkl  — char-level TF-IDF (fitted on training data)
#
# VADER + handcrafted features are rule-based (no fitting needed) so they
# are NOT saved as .pkl — we just call the same functions below.

model_path      = os.path.join(TRAIN_DIR, 'sentiment_model_v3.pkl')
word_tfidf_path = os.path.join(TRAIN_DIR, 'tfidf_word_vectorizer_v3.pkl')
char_tfidf_path = os.path.join(TRAIN_DIR, 'tfidf_char_vectorizer_v3.pkl')

with open(model_path, 'rb') as f:
    model = pickle.load(f)

with open(word_tfidf_path, 'rb') as f:
    word_tfidf = pickle.load(f)

with open(char_tfidf_path, 'rb') as f:
    char_tfidf = pickle.load(f)

HAS_PROBA = hasattr(model, 'predict_proba')

print('✅ Model and Vectorizers loaded!')
print(f'   Model type         : {type(model).__name__}')
print(f'   Confidence scores  : {"Supported ✅" if HAS_PROBA else "Not supported"}')

In [ ]:
# ── Cell 4 — Preprocessing & Feature Functions (Identical to train_3.ipynb) ───
# CRITICAL: Every function here must match train_3 exactly.
# The model was trained on features built by these exact functions.
# Any difference in preprocessing = wrong features = wrong predictions.

# ── 4a: Contraction map (fallback if library not installed) ──
CONTRACTION_MAP = {
    "isn't": "is not", "wasn't": "was not", "weren't": "were not",
    "hasn't": "has not", "haven't": "have not", "hadn't": "had not",
    "doesn't": "does not", "didn't": "did not", "don't": "do not",
    "won't": "will not", "wouldn't": "would not", "couldn't": "could not",
    "shouldn't": "should not", "can't": "cannot", "cannot": "cannot",
    "it's": "it is", "i'm": "i am", "i've": "i have", "i'll": "i will",
    "i'd": "i would", "they're": "they are", "we're": "we are",
    "you're": "you are", "he's": "he is", "she's": "she is",
    "that's": "that is", "there's": "there is", "what's": "what is",
    "let's": "let us", "who's": "who is", "it'll": "it will",
    "they've": "they have", "we've": "we have", "you've": "you have",
    "they'll": "they will", "we'll": "we will", "you'll": "you will",
    "they'd": "they would", "we'd": "we would", "you'd": "you would",
    "n't": " not"
}

# ── 4b: Negation words to KEEP (not remove as stopwords) ──
NEGATION_WORDS = {
    'no', 'not', 'nor', 'never', 'neither', 'nobody', 'nothing', 'nowhere',
    'hardly', 'scarcely', 'barely', 'without', 'cannot'
}

base_stop_words = set(stopwords.words('english'))
stop_words = base_stop_words - NEGATION_WORDS
lemmatizer = WordNetLemmatizer()
vader      = SentimentIntensityAnalyzer()

def expand_contractions(text):
    if HAS_CONTRACTIONS:
        return contractions.fix(text)
    text = text.lower()
    for contraction, expansion in CONTRACTION_MAP.items():
        text = re.sub(re.escape(contraction), expansion, text)
    return text

def preprocess(text):
    """Cleans raw review text — must match train_3 exactly."""
    text = str(text).lower()
    text = expand_contractions(text)                              # NEW in v3
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

def get_vader_features(text):
    """Returns 4 VADER sentiment scores for the RAW review text."""
    scores = vader.polarity_scores(str(text))
    return scores['compound'], scores['pos'], scores['neg'], scores['neu']

def get_handcrafted_features(text):
    """Returns 5 handcrafted features from the RAW review text."""
    text  = str(text)
    words = text.split()
    review_len        = len(words)
    exclamation_cnt   = text.count('!')
    question_cnt      = text.count('?')
    alpha_chars       = [c for c in text if c.isalpha()]
    caps_ratio        = sum(1 for c in alpha_chars if c.isupper()) / (len(alpha_chars) + 1)
    unique_word_ratio = len(set(w.lower() for w in words)) / (len(words) + 1)
    return review_len, exclamation_cnt, question_cnt, caps_ratio, unique_word_ratio

print('✅ All preprocessing & feature functions loaded (matching train_3 pipeline).')

In [ ]:
# ── Cell 5 — Predict Function ─────────────────────────────────────────────────
# Builds the EXACT same feature matrix used during training:
#   [Word TF-IDF | Char TF-IDF | VADER (4 cols) | Handcrafted (5 cols)]

EMOJI_MAP = {
    'Highly Positive' : '🌟 HIGHLY POSITIVE',
    'Positive'        : '😊 POSITIVE',
    'Neutral'         : '😐 NEUTRAL',
    'Negative'        : '😞 NEGATIVE',
    'Highly Negative' : '💀 HIGHLY NEGATIVE'
}

def get_confidence_bar(confidence, width=20):
    filled = int(confidence / 100 * width)
    bar = '█' * filled + '░' * (width - filled)
    return f'[{bar}] {confidence:.1f}%'

def predict_sentiment(review_text):
    """
    Full v3 prediction pipeline:
      1. Preprocess raw text (contraction expansion + cleaning)
      2. Extract Word + Char TF-IDF features
      3. Extract VADER + handcrafted numeric features
      4. Combine into one feature matrix
      5. Predict with loaded model
    """
    # Step 1: Clean text
    cleaned = preprocess(review_text)

    # Step 2: TF-IDF features (use .transform, NOT .fit_transform)
    X_word = word_tfidf.transform([cleaned])
    X_char = char_tfidf.transform([cleaned])

    # Step 3: VADER + handcrafted features from RAW text
    vader_feats = get_vader_features(review_text)
    craft_feats = get_handcrafted_features(review_text)
    extra = np.array([list(vader_feats) + list(craft_feats)])  # shape (1, 9)
    X_extra = csr_matrix(extra)

    # Step 4: Combine — must match training order exactly
    features = hstack([X_word, X_char, X_extra])

    # Step 5: Predict
    label = model.predict(features)[0]
    emoji = EMOJI_MAP.get(label, label)

    if HAS_PROBA:
        proba      = model.predict_proba(features)[0]
        confidence = round(max(proba) * 100, 1)
    else:
        confidence = None

    return label, emoji, confidence, cleaned

print('✅ predict_sentiment() ready.')

In [ ]:
# ── Cell 6 — Single Review Prediction ────────────────────────────────────────
# ✏️  CHANGE THIS REVIEW TO ANYTHING YOU WANT:
my_review = "This movie was absolutely incredible. The performances were outstanding and the story left me in tears."

label, emoji, confidence, cleaned = predict_sentiment(my_review)

print('='*60)
print('   SENTIMENT PREDICTION RESULT  (v3 Model)')
print('='*60)
print(f'  Original Review : {my_review}')
print(f'  Cleaned Text    : {cleaned}')
print(f'  Predicted Class : {label}')
print(f'  Sentiment       : {emoji}')
if confidence is not None:
    print(f'  Confidence      : {get_confidence_bar(confidence)}')
print('='*60)

In [ ]:
# ── Cell 7 — Batch Prediction on Multiple Reviews ─────────────────────────────
test_reviews = [
    "A masterpiece that will be remembered forever. Truly breathtaking.",
    "It was okay. Not great, not terrible. Just an average experience.",
    "Terrible movie. I fell asleep twice and the story made no sense at all.",
    "Loved the chemistry between the actors. A feel-good film perfect for the weekend.",
    "An absolute disaster. Worst film of the year without any doubt.",
    "The cinematography was decent but the plot was too predictable.",
    "One of the most emotionally powerful films I have ever seen in my life.",
    "Nothing special. Very generic and forgettable from start to finish."
]

print('📋 BATCH PREDICTION RESULTS  (v3 Model)')
print('='*70)
batch_results = []
for i, review in enumerate(test_reviews, 1):
    label, emoji, confidence, _ = predict_sentiment(review)
    conf_str = get_confidence_bar(confidence) if confidence is not None else 'N/A'
    batch_results.append({
        'Review'              : review,
        'Predicted Sentiment' : label,
        'Confidence (%)'      : confidence if confidence is not None else 'N/A'
    })
    print(f'  [{i}] {emoji}')
    print(f'      Confidence : {conf_str}')
    print(f'      {review[:70]}...' if len(review) > 70 else f'      {review}')
    print()

batch_df = pd.DataFrame(batch_results)
print('='*70)
print(f'\n✅ {len(test_reviews)} reviews predicted.')
batch_df

In [ ]:
# ── Cell 8 — Save Batch Prediction Results to Drive/predict/ ──────────────────
output_path = os.path.join(PREDICT_DIR, 'prediction_results_v3.csv')
batch_df.to_csv(output_path, index=False)

print(f'✅ Batch prediction results saved to:')
print(f'   {output_path}')
batch_df

In [ ]:
# ── Cell 9 — Interactive Prediction (Type and Predict) ────────────────────────
# Run this cell for an interactive loop.
# Type a review and press Enter to get prediction + confidence.
# Type 'quit' to stop.

print('🎬 Interactive Movie Review Sentiment Predictor  (v3 Model)')
print('   Type a movie review and press Enter.')
print('   Type  quit  to exit.\n')

while True:
    user_input = input('Enter review: ').strip()
    if user_input.lower() == 'quit':
        print('\n👋 Exiting predictor. Goodbye!')
        break
    if not user_input:
        print('  ⚠️  Please enter a review.\n')
        continue
    label, emoji, confidence, _ = predict_sentiment(user_input)
    print(f'  → Sentiment  : {emoji}')
    if confidence is not None:
        print(f'  → Confidence : {get_confidence_bar(confidence)}')
    print()